# Utilities

## Housekeeping

In [2]:
import os
import random
import time
import re

from pathlib import Path
from typing import Literal, Optional, Union

import pandas as pd
from tqdm.notebook import tqdm

In [3]:
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
DATA_DIR = Path('/content/drive/MyDrive/ZBLabSummerSchool2026/data') if IN_COLAB else Path('../data')

## KWIC

In [4]:
### ⬇️⬇️⬇️  Adjust here if you want to continue with your own query
CORPUS_NAME = "armenpflege"
USE_YOUR_DATA = False
### ⬆️⬆️⬆️

In [ ]:
if USE_YOUR_DATA:

    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')

    RAWDATA_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.parquet" # Use data from filtering module

    print(f"Loading raw data ...", end=" ")
    raw_df = pd.read_parquet(RAWDATA_PATH)
    print("Done!")

    LEMMA_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.lemmas.parquet"
    print(f"Loading lemma data ...", end=" ")
    lemma_df = pd.read_parquet(LEMMA_PATH)
    print("Done!")

    TOKENS_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.tokens.parquet"
    print(f"Loading token data ...", end=" ")
    wordform_df = pd.read_parquet(TOKENS_PATH)
    print("Done!")
    
    raw_df = raw_df.join(lemma_df)
    raw_df = raw_df.join(wordform_df)
else:
    RAWDATA_URL = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.parquet"
    print(f"Loading raw data ...", end=" ")
    raw_df = pd.read_parquet(RAWDATA_URL)
    print("Done!")

    LEMMA_URL = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.lemmas.parquet"
    print(f"Loading lemma data ...", end=" ")
    lemma_df = pd.read_parquet(LEMMA_URL)
    print("Done!")

    TOKENS_URL = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.tokens.parquet"
    print(f"Loading token data ...", end=" ")
    wordform_df = pd.read_parquet(TOKENS_URL)
    print("Done!")

    raw_df = raw_df.join(lemma_df)
    raw_df = raw_df.join(wordform_df)

    print(f"Data loaded successfully! {len(raw_df)} rows.")

Loading raw data ... Done!
Loading lemma data ... Done!
Loading token data ... Done!
Data loaded successfully! 2633 rows.


In [6]:
from dataclasses import dataclass
from typing import List, Generator, Tuple
import re


@dataclass
class KWIC:
    left_context: str
    keyword: str
    right_context: str
    doc_id: str


class KWICEngine:
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self._search_df = df  # This will be set to a filtered version of the DataFrame when searching with a year range

    def iterate_kwics(
        self,
        search_term: str,
        column: Literal["lemmas", "tokens"] = "lemmas",
        context_size: int = 15,
    ) -> Generator[KWIC, None, None]:
        regex = re.compile(rf"{search_term}")

        for _, row in self._search_df.iterrows():
            flat_tokens = [token for sentence in row["tokens"] for token in sentence]
            flat_searchspace = [lemma for sentence in row[column] for lemma in sentence]

            for idx, token in enumerate(flat_searchspace):
                if regex.fullmatch(token):
                    left_context = " ".join(
                        flat_tokens[max(0, idx - context_size) : idx]
                    )
                    right_context = " ".join(
                        flat_tokens[idx + 1 : idx + 1 + context_size]
                    )
                    row_index = row.name
                    yield KWIC(left_context, token, right_context, row_index)

    def show_kwics(self, kwics: List[KWIC]):
        if not kwics:
            print("No matches found.")
            return

        max_keyword_width = max(len(kwic.keyword) for kwic in kwics)
        max_left_width = max(len(kwic.left_context) for kwic in kwics) + 2
        max_right_width = max(len(kwic.right_context) for kwic in kwics) + 2

        for kwic in kwics:
            print(
                f"{kwic.left_context:>{max_left_width}}  "
                f"\x1b[1m{kwic.keyword:<{max_keyword_width}}\x1b[0m  "
                f"{kwic.right_context:<{max_right_width}}"
            )

    def show_kwics_strlength(self, kwics: List[KWIC]):
        if not kwics:
            print("No matches found.")
            return

        total_width = 160
        max_keyword_width = max(len(kwic.keyword) for kwic in kwics)
        left_width = total_width // 2 - max_keyword_width // 2 - 2
        right_width = total_width - left_width - max_keyword_width - 4

        for kwic in kwics:
            left_display = (
                kwic.left_context[-left_width:]
                if len(kwic.left_context) > left_width
                else kwic.left_context
            )
            right_display = (
                kwic.right_context[:right_width]
                if len(kwic.right_context) > right_width
                else kwic.right_context
            )
            print(
                f"{kwic.doc_id} - "
                f"{left_display:>{left_width}}  \x1b[1m{kwic.keyword:<{max_keyword_width}}\x1b[0m  {right_display:<{right_width}}"
            )

    def get_kwics(
        self,
        search_term: str,
        column: Literal["lemmas", "tokens"] = "lemmas",
        n: Optional[int] = None,
        context_size: int = 15,
    ) -> List[KWIC]:
        kwic_generator = self.iterate_kwics(search_term, column, context_size)
        if n is None:
            return list(kwic_generator)

        kwics = []
        for _, kwic in zip(range(n), kwic_generator):
            kwics.append(kwic)
        return kwics

    def show_kwics_for_term(
        self,
        search_term: str,
        column: Literal["lemmas", "tokens"] = "lemmas",
        n: Optional[int] = None,
        context_size: Optional[int] = None,
        year_range: Optional[Tuple[int, int]] = None,
    ):
        if year_range is None:
            self._search_df = self.df
        else:
            start, end = year_range
            self._search_df = self.df[
                (self.df["year"] >= start) & (self.df["year"] <= end)
            ]
        if context_size is None:
            kwics = self.get_kwics(search_term, column, n, 30)
            self.show_kwics_strlength(kwics)
        else:
            kwics = self.get_kwics(search_term, column, n, context_size)
            self.show_kwics(kwics)


k = KWICEngine(raw_df)

Search for your terms of interest.

You can use regular expressions in your search terms.

In [ ]:
# ⬇️⬇️⬇️
k.show_kwics_for_term("Not", column="lemmas", n=20)
# k.show_kwics_for_term("Noth", column="lemmas", n=20)
# k.show_kwics_for_term("Noth?", column="lemmas", n=20, year_range=(1850, 1860))
# k.show_kwics_for_term("Bro[dt]", column="lemmas", n=20, context_size=10)
# ⬆️⬆️⬆️

NZZ-1897-05-29-b-i0002 - ote Plakat , welches die Unterschrist von acht Glaßbrenner « trägt , legt die  Not  der Arbeiter dar . welchen der ausgezahlt Lohn nicht wird , weil die Summen 
NZZ-1899-08-15-c-i0005 - ein Einwohnerprinzip , das kein Ganzes ist , sondern viel durchbrochen . Ohne  Not  soll man einen Sprung solchen ins Ungewisse nicht thun . Ein Rückwärts gabe 
NZZ-1899-08-15-c-i0005 - aß einige ältere uud schwerkranke Personen sowie auch Kinder nur mit Mühe und  Not  und im letzten Augenblicke dem schauerlichen Flammentod entrissen werden kon
NZZ-1899-08-15-c-i0005 - erntesten Stocke bei der verwendet werden konnte . iturz , die unverschuldete  Not  und der Jammer ist groß ! In dem betreffenden Viertel wohnten wohl von den ä
NZZ-1899-08-15-c-i0005 - schon jeht den herzlichste » Dank aussprechen . Unterstützt und helft uns dic  Not  lindern , wofür der Allmächtig « den geben möge . Lohn DasHül fs komitee : R
NZZ-1948-02-26-b-i0005 - ausgenommen vielleicht dann , wenn ein solche

## Parquet to human-readable format (CSV/JSON)

⚠️ Lists (as for sentences, tokens and lemmas) are not natively serializable to CSV.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

In [6]:
### ⬇️⬇️⬇️ 💽 Adjust here for the name of the file in your Google Drive
FILENAME = "armenpflege.filtered.lemmas.parquet"
### ⬆️⬆️⬆️

try:
    df = pd.read_parquet(DATA_DIR / FILENAME)
    print(f"Loaded data from {DATA_DIR / FILENAME}")
except Exception as e:
    print(f"Could not load data from {DATA_DIR / FILENAME}. Please check the filename and path. Error: {e}")

Loaded data from ../data/armenpflege.filtered.lemmas.parquet


In [ ]:
try:
    df.to_csv("df.csv")
    print("Saved dataframe to df.csv")
except Exception as e:    print(f"Could not save dataframe to CSV. Error: {e}")
    df.to_json("df.jsonl", orient="records", lines=True)
    print("Saved dataframe to df.jsonl")

KeyboardInterrupt: 